In [1]:
import sys
import os

# Notebook 中没有 __file__，用当前工作目录向上找项目根目录
sys.path.insert(0, os.path.abspath(".."))

from config import chat, chat_stream, get_llm_config, print_config


# ============================================================
# 1. 基础调用 - 最简单的一次性问答
# ============================================================
def basic_chat(prompt: str) -> str:
    """
    最基本的调用方式：发送一条消息，等待完整回复。

    核心概念：
    - 通过 config.chat() 统一接口，自动选择 Ollama 或 Groq 后端
    - messages: 对话历史，格式为 [{"role": "user/assistant/system", "content": "..."}]
    """
    return chat(
        [{"role": "user", "content": prompt}],
    )

In [2]:
basic_chat("你好，简单介绍一下自己")


'<think>\n好的，用户让我简单介绍一下自己。首先，我需要明确自己是什么样的AI。我是由阿里云开发的大模型，英文名是Qwen，中文名通义千问。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本等，还能进行逻辑推理、编程，表达观点，玩游戏等。\n\n接下来要考虑用户的需求是什么。用户可能刚接触我，想了解我的基本功能和应用场景。他们可能希望知道我能帮助他们做什么，比如学习、工作还是娱乐。此外，用户可能希望了解我的技术背景，比如基于什么模型，有什么特点。\n\n然后要组织语言，确保信息准确又简洁。需要提到我的多语言支持、应用场景、技术基础（超大规模参数量），以及持续迭代更新。同时，避免使用过于专业的术语，让解释更友好易懂。最后用轻松的语气邀请用户提问，促进互动。\n\n还需要检查是否遗漏了重要信息，比如我的训练数据截止时间是2024年10月，这个时间点可能影响我知识的时效性，所以需要特别说明。另外，强调我的目标是成为用户可靠的助手，帮助他们解决问题。注意保持语句连贯，信息全面，同时符合中文表达习惯，不使用Markdown格式。\n</think>\n\n你好！我是通义千问，是阿里巴巴集团旗下的超大规模语言模型。我能够回答问题、创作文字，比如写故事、写公文、写邮件、写剧本、逻辑推理、编程等等，还能表达观点，玩游戏等。我支持多种语言，包括但不限于中文、英文、德语、法语、西班牙语等。我的目标是成为一个可靠的人工智能助手，帮助用户解决问题和提供信息。\n\n如果你有任何问题或需要帮助，随时告诉我，我会尽力为你解答！'

In [ ]:
# 显示当前后端配置
print_config()

In [ ]:
def stream_chat(prompt: str):
    """
    流式调用：模型边生成边输出，不用等全部生成完。

    核心概念：
    - 通过 config.chat_stream() 统一接口
    - 返回的是一个迭代器，每次 yield 一小段文本
    - 适合长回答或需要实时显示的场景
    """
    print(f"\n{'='*50}")
    print(f"[流式输出] 问题: {prompt}")
    print(f"{'='*50}")

    full_response = ""
    for token in chat_stream([{"role": "user", "content": prompt}]):
        print(token, end="", flush=True)
        full_response += token

    print()  # 换行
    return full_response


# ============================================================
# 3. 多轮对话 - 维护对话历史
# ============================================================
def multi_turn_demo():
    """
    多轮对话的关键：把之前的 messages 都传给模型。

    核心概念：
    - 模型本身是无状态的，每次调用都是独立的
    - 要实现"记忆"，必须手动维护 messages 列表
    - 每次把完整的对话历史发送给模型
    - messages 越长，消耗的 tokens 越多（注意 context window 限制）
    """
    print(f"\n{'='*50}")
    print("[多轮对话演示]")
    print(f"{'='*50}")

    messages = []

    # 可选：设置 system prompt 定义助手角色
    messages.append(
        {
            "role": "system",
            "content": "你是一个友好的AI助手，回答简洁明了，用中文回答。",
        }
    )

    # 模拟3轮对话
    questions = [
        "Python 的 GIL 是什么？一句话解释",
        "那它对多线程有什么影响？",  # 这个问题依赖上文的"它"
        "有什么替代方案？",  # 继续追问
    ]

    for q in questions:
        print(f"\n用户: {q}")
        messages.append({"role": "user", "content": q})

        answer = chat(messages)

        # 把助手回复也加入历史，这样下一轮模型能看到
        messages.append({"role": "assistant", "content": answer})
        print(f"助手: {answer}")

    print(f"\n[对话历史共 {len(messages)} 条消息]")

In [ ]:
stream_chat("介绍一下 RAG（检索增强生成）的基本原理")

In [ ]:
multi_turn_demo()

In [3]:
# ============================================================
# 5. 调节参数 - Temperature, num_ctx 等
# ============================================================
def parameter_demo():
    """
    常用参数说明：

    - temperature: 控制随机性 (0=确定性, 1=较随机, 2=很随机)
      RAG 任务建议 0.1-0.3，创意任务 0.7+
    - top_p: nucleus sampling，和 temperature 配合使用
    - seed: 设置随机种子，相同 seed 输出一致（可复现）
    """
    print(f"\n{'='*50}")
    print("[Temperature 参数对比]")
    print(f"{'='*50}")

    prompt = "用一句话描述春天"

    for temp in [0.1, 0.7, 1.5]:
        print(f"\ntemperature={temp}:")
        response = chat(
            [{"role": "user", "content": prompt}],
            temperature=temp,
        )
        print(f"  {response}")

In [4]:
        parameter_demo()


[Temperature 参数对比]

temperature=0.1:
  <think>
好的，用户让我用一句话描述春天。首先，我需要确定用户的需求是什么。可能他们需要一句简洁而生动的句子，用于某个特定的场合，比如诗歌、作文，或者作为某个项目的标题。接下来，我得考虑春天的典型元素，比如天气、植物、动物、人们的活动等等。春天通常和温暖、新生、花朵开放、鸟儿归来有关。

然后，我需要把这些元素浓缩成一句话，同时保持语言的优美和流畅。可能需要用一些比喻或者拟人化的手法，让句子更有画面感。比如“春天是大地苏醒的画卷”这样的结构，既简洁又有意境。还要注意句子的节奏和韵律，使其朗朗上口。

另外，用户可能希望这句话有一定的文学性，但又不至于太过复杂。所以需要平衡用词，避免生僻字，同时保持美感。可能还要考虑不同的文化背景，确保描述的元素是普遍认可的，比如樱花、绿草、微风等。

最后，检查有没有更好的词汇替换，比如“苏醒”是否比“醒来”更有力量感，“画卷”是否能准确传达春天的美丽和广阔。确认句子结构是否通顺，有没有语法错误。确保这句话能够准确传达春天的生机与活力，同时满足用户的需求。
</think>

春天是万物复苏的季节，温暖的阳光唤醒沉睡的大地，嫩绿的芽苞在枝头绽放，空气中弥漫着花香与希望。

temperature=0.7:
  <think>
嗯，用户让我用一句话描述春天。首先，我需要确定用户的需求是什么。可能他们需要一句简洁而生动的句子，用来表达春天的特色。春天通常和新生、温暖、花开有关，所以我要把这些元素结合起来。

接下来，我得考虑用哪些词汇比较合适。要避免太普通的词，比如“美丽的春天”，可能不够有新意。可能需要一些比喻或者拟人化的手法，让句子更生动。比如“苏醒”、“绽放”这样的动词，能传达出活力的感觉。

然后，用户可能希望这句话有画面感，所以需要包含视觉元素，比如颜色。绿色、粉色、嫩芽、花朵这些意象比较适合。同时，春天也常与希望、生机联系在一起，可以加入这些抽象概念，让句子更有深度。

还要注意句子的节奏和结构，可能用对仗或者排比会更好。比如前半句描述自然景象，后半句表达情感或抽象概念。例如“万物复苏，生机盎然；芳菲吐艳，希望萌芽。”这样既有画面感，又有层次。

不过用户要求是一句话，所以需要更简洁。可能需要用分号或者逗号连接，保持连贯。同时要避免太复杂的